# Data Science for Business - Multilayer Perceptron (MLP) on Wine Reviews with PyTorch

*Session 10 · ISLP 10.4 Document Classification · 10.5.1 (word embeddings) · 10.7.2 (early stopping)*

Sessions 8–10 all use PyTorch, the framework also used in ISLP Chapter 10. As in Sessions 8 and 9, we write the training loop by hand.

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
# Install packages that are not pre-installed on Colab
# !pip install torchinfo

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

Use a GPU if one is available: an NVIDIA GPU (`cuda`, e.g., on Colab), Apple silicon (`mps`), or else the CPU.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Using device:', device)

## Problem description

The dataset contains 130,000+ wine reviews (Source: https://www.kaggle.com/datasets/zynicide/wine-reviews) written by professional sommeliers. Besides a text, it contains information about the wine, such as the country of origin, the variety, the winery, the price, and the number of points given by the reviewer. The goal of this notebook is to predict the number of points given by the reviewer (the response) based on the text of the review (the predictor). This is a regression problem, so we use the mean absolute error (MAE) as loss.

## Load data

Load data from CSV file.

In [ ]:
corpus = pd.read_csv("https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_10/winemag-data-130k-v2.csv")
corpus.head()

## Prepare data

### Split data

Split data into three sets: training, validation, and test (ISLP 5.1.1). To speed up training, we only use a random sample of 10,000 reviews for training.

In [ ]:
training = corpus.iloc[0:80000,].sample(n=10000, random_state=42) # to speed up training
validation = corpus.iloc[80000:100000,]
test = corpus.iloc[100000:,]

For each dataset, store predictors (the review texts, as a list of strings) and response (the points, as a NumPy array) in separate variables.

In [ ]:
train_texts = training["description"].tolist()
train_points = training["points"].to_numpy(dtype=np.float32)
val_texts = validation["description"].tolist()
val_points = validation["points"].to_numpy(dtype=np.float32)
test_texts = test["description"].tolist()
test_points = test["points"].to_numpy(dtype=np.float32)

Look at the first review of the training set and its points.

In [ ]:
print(train_texts[0])
print('Points:', train_points[0])

### From text to numbers

A neural network can only process numbers, not text. We turn each review into a sequence of integers in three steps:

1. **Tokenization:** split the text into words (*tokens*).
2. **Vocabulary:** keep the 10,000 most frequent words of the training data, as in ISLP 10.4, and replace each word by its index in this vocabulary.
3. **Padding and truncation:** bring all reviews to the same length, so that a minibatch of reviews fits into one tensor.

For steps 1 and 2 we use scikit-learn's `CountVectorizer`. It is normally used to build the bag-of-words representation of ISLP 10.4 (one column per word, counting how often the word occurs in a document). Here we only need two things it learns along the way: the vocabulary and the tokenizer. As always, we fit it on the **training** texts only.

In [ ]:
max_tokens = 10000   # size of the vocabulary

vectorizer = CountVectorizer(max_features=max_tokens, lowercase=True)
vectorizer.fit(train_texts)

`build_analyzer()` returns the tokenizer the vectorizer uses: it lowercases a text and splits it into words. Punctuation is dropped, and so are one-letter words such as "a".

In [ ]:
analyzer = vectorizer.build_analyzer()
print(analyzer(train_texts[0]))

`vocabulary_` is a dictionary that maps each of the 10,000 words to an index between 0 and 9,999 (in alphabetical order). `get_feature_names_out()` lists the words; here we show every 1,000th word.

In [ ]:
vocabulary = vectorizer.vocabulary_
print(len(vocabulary))
print(vectorizer.get_feature_names_out()[::1000])

Now we replace every word of every review by an integer. We reserve two special indices:

- **0** for padding (see below),
- **1** for unknown words, i.e., words that are not in the vocabulary (rare words, or words that only occur in the validation or test data).

The words of the vocabulary therefore get the indices 2 to 10,001 (their vocabulary index + 2). The nested list comprehension reads: for each text, split it into words with the analyzer, and for each word, look up its index. (`int()` turns the NumPy integers stored in `vocabulary_` into plain Python integers.)

In [ ]:
train_ids = [[int(vocabulary[word]) + 2 if word in vocabulary else 1 for word in analyzer(text)] for text in train_texts]
val_ids = [[int(vocabulary[word]) + 2 if word in vocabulary else 1 for word in analyzer(text)] for text in val_texts]
test_ids = [[int(vocabulary[word]) + 2 if word in vocabulary else 1 for word in analyzer(text)] for text in test_texts]

print(train_ids[0])

The reviews have different lengths.

In [ ]:
train_lengths = [len(ids) for ids in train_ids]
print('Average number of words:', np.mean(train_lengths))
print('Maximum number of words:', np.max(train_lengths))
print('Share of reviews with more than 100 words:', np.mean(np.array(train_lengths) > 100))

We bring all reviews to the same length `max_length = 100`, as ISLP 10.5.1 does with $L = 500$ for the (much longer) IMDb reviews:

- Longer reviews are **truncated**: we keep the *last* 100 words (`ids[-max_length:]`). Almost no wine review is that long.
- Shorter reviews are **padded with zeros at the front**, as in ISLP 10.5.1. The real words then always come last, so the last position of every sequence is a real word. This matters for the RNN in the other notebook, which uses its hidden state after the last position; for the average of the word vectors we use here, it makes no difference.

In [ ]:
max_length = 100

train_truncated = [ids[-max_length:] for ids in train_ids]                                   # keep the last max_length words
train_sequences = np.array([[0] * (max_length - len(ids)) + ids for ids in train_truncated])  # pad with zeros at the front
val_truncated = [ids[-max_length:] for ids in val_ids]
val_sequences = np.array([[0] * (max_length - len(ids)) + ids for ids in val_truncated])
test_truncated = [ids[-max_length:] for ids in test_ids]
test_sequences = np.array([[0] * (max_length - len(ids)) + ids for ids in test_truncated])

print(train_sequences.shape, val_sequences.shape, test_sequences.shape)

Let's look at our example review before and after the transformation: the original text, the padded sequence of 100 integers, and the words behind the integers (we invert the vocabulary to look them up).

In [ ]:
index_to_word = {index + 2: word for word, index in vocabulary.items()}
index_to_word[0] = '<pad>'
index_to_word[1] = '<unknown>'

print(train_texts[0])
print(train_sequences[0])
print([index_to_word[index] for index in train_sequences[0]])

### Data loaders

As in Sessions 8 and 9, we turn the data into PyTorch tensors. The word indices become integers (`torch.long`), because the embedding layer looks up rows by index; the points get an extra dimension (`unsqueeze(1)`), because the network outputs one column.

In [ ]:
X_train_tensor = torch.tensor(train_sequences, dtype=torch.long)
X_val_tensor = torch.tensor(val_sequences, dtype=torch.long)
X_test_tensor = torch.tensor(test_sequences, dtype=torch.long)
y_train_tensor = torch.tensor(train_points).unsqueeze(1)
y_val_tensor = torch.tensor(val_points).unsqueeze(1)
y_test_tensor = torch.tensor(test_points).unsqueeze(1)

The data loaders serve minibatches of 128 reviews, so stochastic gradient descent uses minibatches of 128 (ISLP 10.7.2). The training data is reshuffled in every epoch (with a seeded `generator`, so that the results are reproducible); the validation and test data are only predicted, so their order does not matter. We also predict them in minibatches, because all 20,000 or 30,000 reviews at once would need a lot of memory.

In [ ]:
batch_size = 128

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size,
                          shuffle=True, generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

print('Minibatches per epoch:', len(train_loader))

## Train model

### Network architecture

We specify the network as a class that inherits from `nn.Module` (as in Sessions 8 and 9):

- The embedding layer is a trainable lookup table that maps each word index to a vector of length `embedding_dim` (ISLP 10.5.1). A review of 100 indices becomes 100 word vectors.
- We have to reduce these 100 vectors to a single vector per review. `nn.EmbeddingBag(..., mode='mean', padding_idx=0)` does the lookup and averages the word vectors of a review in one step, ignoring the padding. Averaging ignores word order ("not good" = "good, not"), like the bag-of-words model in ISLP 10.4. The book's bag-of-words is a binary 10,000-dimensional vector per document; averaging learned embeddings is a dense cousin of that, not the same model.
- A hidden layer with 32 hidden units and ReLU activation, and a linear output unit for the predicted points.

In [ ]:
class WineMLP(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.embedding = nn.EmbeddingBag(num_embeddings, embedding_dim, mode='mean', padding_idx=0)
        self.hidden = nn.Linear(embedding_dim, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.embedding(x)            # (batch, embedding_dim): average word vector of each review
        x = self.relu(self.hidden(x))
        x = self.output(x)
        return x

Let's create an instance of the network. The embedding layer needs one row for each of the 10,000 words plus the two special indices (padding and unknown).

We initialize the bias of the output unit with the average points in the training data (about 88). Otherwise, the network would start by predicting values around 0 and would spend many epochs just moving its predictions up to the right range.

We also change the starting values of the word vectors. By default, PyTorch draws them from a standard normal distribution, i.e., values of about ±1 in each of the 300 dimensions. Training works much better here if all word vectors start close to zero, with small random values between −0.05 and 0.05 (the vector of the padding index 0 stays zero).

ISLP 10.5.1 uses an embedding size of $m = 32$ and notes that the embedding matrix is by far the biggest cost. With `embedding_dim = 300` (the size of popular pretrained embeddings such as GloVe), our embedding layer alone has about $10{,}000 \cdot 300 = 3$ million parameters.

In [ ]:
embedding_dim = 300
num_embeddings = max_tokens + 2   # 10,000 words + padding + unknown
mean_points = train_points.mean()

model = WineMLP(num_embeddings, embedding_dim)
model.embedding.weight.data.uniform_(-0.05, 0.05)  # small random starting values for the word vectors
model.embedding.weight.data[0] = 0                 # the padding vector stays zero
model.output.bias.data.fill_(float(mean_points))   # start the output unit at the average rating
model = model.to(device)

`torchinfo.summary` shows the output shape and number of parameters of each layer for a minibatch of 128 reviews. The input consists of word indices, so we pass `dtypes=[torch.long]`.

In [ ]:
summary(model, input_size=(batch_size, max_length), dtypes=[torch.long], device=device)

### Training loop

We minimize the mean absolute error (`nn.L1Loss`), the error measure ISLP 10.6 uses to compare models on the `Hitters` data. The optimizer is Adam, a variant of stochastic gradient descent, with learning rate ρ = 0.001 (`lr`).

In [ ]:
criterion = nn.L1Loss()   # mean absolute error
optimizer = optim.Adam(model.parameters(), lr=0.001)

The training loop is the one you know from Sessions 8 and 9: the outer loop runs over epochs, the inner loop over minibatches (reset gradients, forward pass, loss, backward pass, update weights), and after each epoch we compute the validation MAE in evaluation mode without gradients.

New is **early stopping** (ISLP 10.7.2), written out at the end of each epoch:

- If the validation MAE is the best so far, we store a copy of the current weights. `model.state_dict()` returns the weights themselves, which keep changing during training, so we need `copy.deepcopy` to freeze a snapshot.
- If the validation MAE has not improved for `patience = 2` epochs in a row, we stop training (`break`). We train for at most 10 epochs.
- After training, we restore the best weights with `model.load_state_dict()`.

We use the validation set for early stopping, never the test set.

In [ ]:
epochs = 10
patience = 2
train_maes = []   # average training MAE per epoch
val_maes = []     # validation MAE per epoch
best_val_mae = float('inf')
epochs_without_improvement = 0

for epoch in range(epochs):                                   # outer loop: epochs
    # Training
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:                     # inner loop: minibatches of 128 reviews
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()                                 # reset gradients
        predictions = model(batch_X)                          # forward pass
        loss = criterion(predictions, batch_y)                # MAE on this minibatch
        loss.backward()                                       # backward pass
        optimizer.step()                                      # update weights
        epoch_loss += loss.item()
    train_maes.append(epoch_loss / len(train_loader))

    # Validation
    model.eval()
    val_abs_error = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            val_abs_error += (model(batch_X) - batch_y).abs().sum().item()   # sum of absolute errors
    val_maes.append(val_abs_error / len(val_points))
    print(f'Epoch {epoch + 1}/{epochs}, Training MAE: {train_maes[-1]:.4f}, Validation MAE: {val_maes[-1]:.4f}')

    # Early stopping
    if val_maes[-1] < best_val_mae:                           # new best model: keep a copy of its weights
        best_val_mae = val_maes[-1]
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print('Early stopping: no improvement for', patience, 'epochs')
            break

model.load_state_dict(best_weights)                           # restore the weights of the best epoch
print('Restored the weights of epoch', val_maes.index(best_val_mae) + 1, 'with validation MAE', round(best_val_mae, 4))

Plot the learning process.

In [ ]:
plt.plot(range(1, len(train_maes) + 1), train_maes, label='Training')
plt.plot(range(1, len(val_maes) + 1), val_maes, label='Validation')
plt.title('Model MAE')
plt.xlabel('Epoch')
plt.ylabel('Mean absolute error')
plt.legend()
plt.show()

A validation MAE that rises while the training MAE keeps falling indicates overfitting; this is where early stopping kicks in. (The training MAE is averaged over the minibatches of an epoch while the weights are still changing, so it lags a bit behind.)

## Make predictions

The model now has the weights of the best epoch. Make predictions on the test set, minibatch by minibatch, in evaluation mode and without gradients. `.cpu()` moves the predictions back from the GPU, and `torch.cat` puts the minibatches together.

In [ ]:
model.eval()
test_preds = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        test_preds.append(model(batch_X.to(device)).cpu())
test_preds = torch.cat(test_preds).numpy().ravel()

Calculate the prediction error (MAE) on the test set.

In [ ]:
print('Test MAE:', mean_absolute_error(test_points, test_preds))

Is this good? As a naive baseline, we always predict the mean points of the training set, no matter what the review says.

In [ ]:
baseline_preds = np.full(len(test_points), train_points.mean())
print('Baseline MAE:', mean_absolute_error(test_points, baseline_preds))

Let's look at a few reviews of the test set with their true and predicted points.

In [ ]:
pd.DataFrame({'review': test_texts[:5],
              'true points': test_points[:5],
              'predicted points': test_preds[:5].astype(float).round(1)})

The network clearly beats the naive baseline, so the review texts contain a lot of information about the points.

## Your turn!

**Your turn!** Change the embedding size to 32 (as in ISLP 10.5.1) or the maximum sequence length. Create a new model and optimizer and rerun the training loop. How do the number of parameters, the test MAE and the training time change?

In [ ]:
# YOUR CODE HERE